# Data Preprocessing Pipeline
> **Project:** AI-Based Product Demand Forecasting System  
> **Objective:** Transform raw Online Retail data into a clean, analysis-ready dataset.

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

plt.rcParams['figure.figsize'] = (12, 4)
sns.set_style('whitegrid')
print('Environment ready.')

In [ ]:
# ── Load raw data ────────────────────────────────────────────────
RAW_PATH = '../notebook/data/data.csv'
PROCESSED_PATH = '../notebook/data/processed_data.csv'

df_raw = pd.read_csv(RAW_PATH, encoding='latin1')
print(f'Raw dataset: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
df_raw.head(3)

In [ ]:
# ── Step 1: Parse dates ──────────────────────────────────────────
df = df_raw.copy()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], infer_datetime_format=True)

print('Date range after parsing:')
print(f'  Min: {df["InvoiceDate"].min()}')
print(f'  Max: {df["InvoiceDate"].max()}')
print(f'  DType: {df["InvoiceDate"].dtype}')

In [ ]:
# ── Step 2: Filter valid transactions ───────────────────────────
initial_rows = len(df)

# Remove cancellations (InvoiceNo starts with 'C')
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
after_cancel = len(df)

# Remove non-positive quantities and prices
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
after_negatives = len(df)

# Remove test / non-product stock codes
non_product = ['POST', 'D', 'M', 'BANK CHARGES', 'PADS', 'DOT']
df = df[~df['StockCode'].astype(str).isin(non_product)]
after_nonprod = len(df)

print(f'Initial rows            : {initial_rows:,}')
print(f'After removing cancels  : {after_cancel:,}  (removed {initial_rows - after_cancel:,})')
print(f'After removing negatives: {after_negatives:,}  (removed {after_cancel - after_negatives:,})')
print(f'After removing non-prod : {after_nonprod:,}  (removed {after_negatives - after_nonprod:,})')

In [ ]:
# ── Step 3: Handle missing values ───────────────────────────────
print('Missing values before treatment:')
print(df.isnull().sum())

# Drop rows with no Description (very few)
df.dropna(subset=['Description'], inplace=True)

# Fill missing CustomerID with placeholder -1
df['CustomerID'] = df['CustomerID'].fillna(-1).astype(int)

print('\nMissing values after treatment:')
print(df.isnull().sum())

In [ ]:
# ── Step 4: Remove outliers via IQR method ──────────────────────
def remove_outliers_iqr(series, multiplier=3.0):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - multiplier * IQR, Q3 + multiplier * IQR
    return series.between(lower, upper)

qty_mask = remove_outliers_iqr(df['Quantity'])
price_mask = remove_outliers_iqr(df['UnitPrice'])

before = len(df)
df = df[qty_mask & price_mask]
print(f'Rows removed as outliers: {before - len(df):,}')
print(f'Rows remaining          : {len(df):,}')

In [ ]:
# ── Step 5: Extract date/time features ──────────────────────────
df['Year']        = df['InvoiceDate'].dt.year
df['Month']       = df['InvoiceDate'].dt.month
df['Day']         = df['InvoiceDate'].dt.day
df['Hour']        = df['InvoiceDate'].dt.hour
df['DayOfWeek']   = df['InvoiceDate'].dt.dayofweek   # 0=Monday
df['WeekOfYear']  = df['InvoiceDate'].dt.isocalendar().week.astype(int)
df['Quarter']     = df['InvoiceDate'].dt.quarter
df['IsWeekend']   = df['DayOfWeek'].isin([5, 6]).astype(int)

# Revenue column
df['Revenue'] = df['Quantity'] * df['UnitPrice']

print('New features added:')
print(df[['Year','Month','Day','Hour','DayOfWeek','WeekOfYear','Quarter','IsWeekend','Revenue']].head(3))

In [ ]:
# ── Step 6: Encode categorical columns ──────────────────────────
le_country = LabelEncoder()
df['CountryCode'] = le_country.fit_transform(df['Country'])

# Top-N stock code encoding (frequency encoding)
top_n = 100
top_stocks = df['StockCode'].value_counts().head(top_n).index
df['StockCodeGroup'] = df['StockCode'].apply(lambda x: x if x in top_stocks else 'OTHER')

print(f'Unique countries encoded : {df["CountryCode"].nunique()}')
print(f'StockCodeGroup categories: {df["StockCodeGroup"].nunique()}')

In [ ]:
# ── Save processed dataset ──────────────────────────────────────
os.makedirs(os.path.dirname(PROCESSED_PATH), exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f'Processed data saved to: {PROCESSED_PATH}')
print(f'Final shape: {df.shape}')

In [ ]:
# ── Summary statistics of processed data ────────────────────────
print('=== Processed Dataset Summary ===')
print(f'Total transactions : {len(df):,}')
print(f'Unique invoices    : {df["InvoiceNo"].nunique():,}')
print(f'Unique products    : {df["StockCode"].nunique():,}')
print(f'Unique customers   : {df["CustomerID"].nunique():,}')
print(f'Unique countries   : {df["Country"].nunique():,}')
print(f'\nRevenue statistics:')
print(df['Revenue'].describe().round(2))

# Revenue distribution
df['Revenue'].clip(upper=200).hist(bins=60, color='steelblue', edgecolor='white')
plt.title('Revenue per Transaction (clipped at £200)')
plt.xlabel('Revenue (£)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## Preprocessing Conclusions

| Step | Action | Impact |
|------|--------|--------|
| Date parsing | Converted InvoiceDate to datetime | Enables time-based features |
| Cancellations | Removed ~2 % of rows | Prevents negative demand |
| Negatives/zeros | Removed invalid qty/price | Data integrity |
| Missing values | Filled CustomerID; dropped bad desc. | No null values remain |
| Outlier removal | IQR × 3 on Qty & Price | ~1-2 % rows removed |
| Date features | Year/Month/Day/Hour/DOW/Quarter | Enables ML models |
| Encoding | LabelEncoder for Country | Numeric input for models |

> **Next step:** Build time-series feature engineering (Notebook 03).